# Generative Modelling II: executable DDPM companion

CPU exercises use the same one-based schedule and fixed posterior variance as the lecture. Optional full training is separate. Normalisation is uint8/127.5 - 1. All small-number fixtures are analytical; recorded image predictions come from the documented model. Run from this notebook directory or the lecture root.

In [ ]:
from pathlib import Path
import sys, torch, numpy as np
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if not (ROOT/'scripts/train_mnist_ddpm.py').exists():
    ROOT = ROOT/'generative-modelling-ii'
sys.path.insert(0, str(ROOT/'scripts'))
from train_mnist_ddpm import make_schedule, extract, TinyDenoiser, training_loss, reverse_step, sample_ddpm
torch.set_num_threads(4)
torch.manual_seed(667)
print('Torch', torch.__version__)

## 1. A schedule with a real clean sentinel
The endpoint must have little signal remaining before starting a sampler from a standard Gaussian is sensible.

In [ ]:
beta, alpha, abar, posterior_var = make_schedule()
assert abar[0] == 1 and posterior_var[1] == 0
assert torch.all(abar[1:] < abar[:-1])
print('arrays:', len(beta), 'terminal retention:', abar[-1].item())

## 2. Manufacture a training tuple
A batch may contain different timesteps. This numerical exercise uses a small authored image tensor, not an MNIST sample.

In [ ]:
x0 = torch.linspace(-1, 1, 3*28*28).reshape(3,1,28,28)
t = torch.tensor([7,350,920], dtype=torch.long)
eps = torch.randn_like(x0)
a = extract(abar,t,x0)
xt = a.sqrt()*x0 + (1-a).sqrt()*eps
print('coefficients', tuple(a.shape), 'input', tuple(xt.shape), 'target', tuple(eps.shape))

## 3. Inspect the named-layer denoiser and gradients
One CPU forward/backward batch checks execution; it does not train a useful model.

In [ ]:
model = TinyDenoiser()
prediction = model(xt,t)
loss = torch.nn.functional.mse_loss(prediction, eps)
loss.backward()
assert prediction.shape == x0.shape
grad = sum(p.grad.abs().sum() for p in model.parameters() if p.grad is not None)
assert grad > 0 and torch.isfinite(loss)
print('parameters', sum(p.numel() for p in model.parameters()), 'loss', loss.item(), 'gradient L1', grad.item())

## 4. The same scalar in forward, loss and reverse calculations

In [ ]:
import math
x2 = .8*1 + .6*(-.5)
eps_hat = -.4
xhat = (x2-.6*eps_hat)/.8
mean = (x2-(.2/.6)*eps_hat)/math.sqrt(.8)
variance = .2*(1-.8)/(1-.64)
sample = mean+math.sqrt(variance)*.3
print(dict(x2=x2, loss=(-.5-eps_hat)**2, xhat=xhat, mean=mean, variance=variance, sample=sample))
assert abs(sample-.8080881928749334)<1e-12

## 5. The final DDPM update ignores fresh sampling noise

In [ ]:
schedule=make_schedule()
x=torch.randn(2,1,28,28); prediction=torch.randn_like(x)
a=reverse_step(x,prediction,1,schedule,torch.zeros_like(x))
b=reverse_step(x,prediction,1,schedule,torch.full_like(x,999))
assert torch.equal(a,b)
print('No extra noise at t=1 verified')

## 6. Cross-attention with hand-designed features
The row axis enumerates image queries. Each row normalises over text keys.

In [ ]:
Q=np.eye(2); K=np.array([[1,0],[0,1],[1,1]]); V=np.array([[1,0],[0,2],[1,1]])
scores=Q@K.T/np.sqrt(2)
A=np.exp(scores-scores.max(axis=1,keepdims=True)); A/=A.sum(axis=1,keepdims=True)
assert np.allclose(A.sum(axis=1),1)
print('attention:',A,'output:',A@V,sep='\n')

## 7. The exact known-mixture score
This is an analytical calculation, not a learned neural score.

In [ ]:
centres=np.stack([2*np.cos(np.arange(8)*np.pi/4),2*np.sin(np.arange(8)*np.pi/4)],1)
x=np.array([[1.,.5],[-1.,0.]])
at=float(abar[350]); means=np.sqrt(at)*centres; v=at*.18**2+1-at
delta=means[None]-x[:,None]
logits=-np.sum(delta**2,axis=-1)/(2*v)
r=np.exp(logits-logits.max(axis=1,keepdims=True)); r/=r.sum(axis=1,keepdims=True)
score=(r[:,:,None]*delta).sum(axis=1)/v
print('score',score,'oracle noise',-np.sqrt(1-at)*score,sep='\n')

## 8. Guidance and masked-token loss are different operations

In [ ]:
eu=np.array([.2,-.3]); ec=np.array([.6,.1])
for s in [0,1,3]: print('guidance',s,eu+s*(ec-eu))
mask=np.array([False,True,False,False,True]); p_true=np.array([.9,.7,.8,.9,.6]); t=.5
ce_sum=-np.log(p_true[mask]).sum()
print('illustrative CE sum',ce_sum,'time-weighted sum',ce_sum/t)
# No division by sequence length in this displayed time-weighted convention.

## 9. Inspect the actual saved experiment
This is a record of training already performed, not a claim that this notebook reruns it.

In [ ]:
import json,hashlib
meta=json.loads((ROOT/'assets/new-model-provenance.json').read_text())
print({k:meta[k] for k in ['steps','batch_size','parameters','seed','alpha_bar_T','sampler']})
checkpoint=ROOT/'models/model.pt'
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest()==meta['checkpoint_sha256']
print('Checkpoint hash verified')

## Optional model training and sampling
Run these commands in the lecture folder. The default device is CPU; use `--device mps` or `--device cuda` only when available. No expensive training occurs automatically in this notebook.

```sh
python scripts/train_mnist_ddpm.py --mode smoke
python scripts/train_mnist_ddpm.py --mode train --steps 2000 --device mps
python scripts/train_mnist_ddpm.py --mode sample --checkpoint models/model.pt --device mps
python scripts/export_evidence.py
```

The final trajectory arrays distinguish chain states from clean estimates. Display clips are not training or sampler clamps. The sampler has no arbitrary timestep skipping. Advanced image and language systems are sourced explanations, not models trained here.